<a href="https://colab.research.google.com/github/AngelGarcia0905/IA/blob/main/PIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generador Markov N-5 para ESP32-S3
### Produce `markov_latam.h` y `markov_usa.h` listos para Arduino IDE

**Orden N-5:** prefijo de 4 chars predice el 5to. Cascada T5->T4->T3->T2.
Tablas ordenadas lexicograficamente para busqueda binaria O(log n) en el ESP32.

Corre cada celda en orden. Los .h quedan en `/content/` para descargar.

---
## Celda 1 — Instalacion de dependencias

In [ ]:
# Hugging Face 'datasets' descarga Wikipedia en streaming (sin bajar los dumps completos de GBs)
!pip install -q datasets tqdm

import re, unicodedata, os
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

print('OK - dependencias listas')

OK - dependencias listas


---
## Celda 2 — Configuracion de parametros
Ajusta aqui antes de correr el resto.

In [ ]:
# ------------------------------------------------------------
# PARAMETROS AJUSTABLES
# ------------------------------------------------------------

# Articulos de Wikipedia por idioma.
# Mas articulos = mejor modelo, mas tiempo de descarga.
# Recomendado Colab gratuito: 8000 por idioma (~15 min total)
NUM_ARTICULOS_ES = 8000
NUM_ARTICULOS_EN = 8000

# Frecuencia minima para incluir un n-grama en la tabla.
# Bajar estos valores genera tablas mas grandes (mas Flash).
# Subirlos hace tablas mas pequenas pero menos cobertura.
MIN_FREQ_T5 = 3   # prefijo 4 chars
MIN_FREQ_T4 = 4   # prefijo 3 chars
MIN_FREQ_T3 = 5   # prefijo 2 chars
MIN_FREQ_T2 = 6   # prefijo 1 char

# Rutas de salida
OUT_LATAM = '/content/markov_latam.h'
OUT_USA   = '/content/markov_usa.h'

print('OK - configuracion cargada')
print(f'  Articulos ES: {NUM_ARTICULOS_ES} | EN: {NUM_ARTICULOS_EN}')
print(f'  Freq minima T5/T4/T3/T2: {MIN_FREQ_T5}/{MIN_FREQ_T4}/{MIN_FREQ_T3}/{MIN_FREQ_T2}')

OK - configuracion cargada
  Articulos ES: 8000 | EN: 8000
  Freq minima T5/T4/T3/T2: 3/4/5/6


---
## Celda 3 — Descarga corpus Wikipedia ES
Dump oficial Wikipedia en espanol via Hugging Face. 100% espanol, sin mezcla de otros idiomas.

In [ ]:
from datasets import load_dataset

print('Descargando Wikipedia ES (primera vez tarda ~5 min, luego queda en cache)...')

# streaming=True evita descargar los ~4GB completos del dump
ds_es = load_dataset(
    'wikimedia/wikipedia', '20231101.es',
    split='train', streaming=True, trust_remote_code=True
)

textos_es = []
for i, art in enumerate(tqdm(ds_es, total=NUM_ARTICULOS_ES, desc='Wikipedia ES')):
    if i >= NUM_ARTICULOS_ES:
        break
    textos_es.append(art['text'])

corpus_es_raw = ' '.join(textos_es)
del textos_es
print(f'OK - {len(corpus_es_raw):,} caracteres descargados')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Descargando Wikipedia ES (primera vez tarda ~5 min, luego queda en cache)...


Wikipedia ES:   0%|          | 0/8000 [00:00<?, ?it/s]

OK - 135,629,053 caracteres descargados


---
## Celda 4 — Descarga corpus Wikipedia EN
Mismo proceso para ingles US. Dump separado, sin mezcla con espanol.

In [ ]:
print('Descargando Wikipedia EN...')

ds_en = load_dataset(
    'wikimedia/wikipedia', '20231101.en',
    split='train', streaming=True, trust_remote_code=True
)

textos_en = []
for i, art in enumerate(tqdm(ds_en, total=NUM_ARTICULOS_EN, desc='Wikipedia EN')):
    if i >= NUM_ARTICULOS_EN:
        break
    textos_en.append(art['text'])

corpus_en_raw = ' '.join(textos_en)
del textos_en
print(f'OK - {len(corpus_en_raw):,} caracteres descargados')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Descargando Wikipedia EN...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia EN:   0%|          | 0/8000 [00:00<?, ?it/s]

OK - 83,640,476 caracteres descargados


---
## Celda 5 — Limpieza y normalizacion
- Todo a **MAYUSCULAS** (el ESP32 trabaja en mayusculas)
- Espanol: elimina acentos (a con acento->A) pero **conserva N con tilde**
- Ingles: ASCII puro A-Z
- Elimina numeros, puntuacion, URLs, referencias tipo [1], etc.

In [ ]:
def limpiar_es(texto):
    """
    Normalizacion para espanol latinoamericano:
    - Mayusculas
    - Elimina acentos en vocales pero conserva N con tilde
    - Solo A-Z + N con tilde + espacio
    """
    texto = texto.upper()
    # Reemplazar vocales acentuadas antes de cualquier otra transformacion
    for acentuada, base in [('A\u0301','A'),('E\u0301','E'),('I\u0301','I'),
                             ('O\u0301','O'),('U\u0301','U'),
                             ('\u00C1','A'),('\u00C9','E'),('\u00CD','I'),
                             ('\u00D3','O'),('\u00DA','U'),('\u00DC','U')]:
        texto = texto.replace(acentuada, base)
    # Solo letras validas + espacio (N con tilde = \u00D1)
    texto = re.sub(r'[^A-Z\u00D1 ]', ' ', texto)
    texto = re.sub(r' +', ' ', texto).strip()
    return texto


def limpiar_en(texto):
    """
    Normalizacion para ingles US:
    - Mayusculas
    - ASCII puro: elimina cualquier caracter fuera de A-Z
    """
    texto = texto.upper()
    # Descomponer Unicode y quedarse solo con ASCII
    texto = unicodedata.normalize('NFD', texto)
    texto = texto.encode('ascii', 'ignore').decode('ascii')
    texto = re.sub(r'[^A-Z ]', ' ', texto)
    texto = re.sub(r' +', ' ', texto).strip()
    return texto


print('Limpiando corpus ES...')
corpus_es = limpiar_es(corpus_es_raw)
del corpus_es_raw
print(f'  ES limpio: {len(corpus_es):,} chars')

print('Limpiando corpus EN...')
corpus_en = limpiar_en(corpus_en_raw)
del corpus_en_raw
print(f'  EN limpio: {len(corpus_en):,} chars')

print()
print('Muestra ES:', corpus_es[:150])
print('Muestra EN:', corpus_en[:150])

Limpiando corpus ES...
  ES limpio: 127,399,543 chars
Limpiando corpus EN...
  EN limpio: 78,754,300 chars

Muestra ES: ANDORRA OFICIALMENTE PRINCIPADO DE ANDORRA ES UN MICRO ESTADO SOBERANO SIN LITORAL UBICADO EN EL SUROESTE DE EUROPA ENTRE ESPAÑA Y FRANCIA EN EL LIMIT
Muestra EN: ANARCHISM IS A POLITICAL PHILOSOPHY AND MOVEMENT THAT IS SKEPTICAL OF ALL JUSTIFICATIONS FOR AUTHORITY AND SEEKS TO ABOLISH THE INSTITUTIONS IT CLAIMS


---
## Celda 6 — Funcion central: construir tabla Markov
Para cada longitud de prefijo (1,2,3,4 chars) desliza una ventana
sobre todo el corpus, cuenta sucesores y elige el mas frecuente.
El resultado se ordena lexicograficamente para busqueda binaria.

In [ ]:
# ============================================================
# CELDA 6 CORREGIDA — reemplaza la celda 6 del notebook
# Cambio clave: ahora SI indexa prefijos que contienen espacios
# al inicio, para cubrir el inicio de palabra (ej: "   H" -> "O")
# ============================================================

def construir_tabla(corpus, largo_prefijo, min_freq):
    """
    Construye tabla Markov de orden (largo_prefijo + 1).

    Parametros:
        corpus        : texto limpio en mayusculas
        largo_prefijo : longitud del prefijo (1=bigrama ... 4=pentagrama)
        min_freq      : frecuencia minima para incluir una entrada

    Retorna:
        dict ordenado { prefijo_str : mejor_sucesor_char }
        ORDENADO LEXICOGRAFICAMENTE para busqueda binaria en ESP32.

    Nota sobre espacios:
        Se permite espacio al INICIO del prefijo (inicio de palabra)
        pero NO como sucesor. Asi el modelo aprende:
          "   H" -> "O"  (inicio absoluto: _H -> O)
          "  HO" -> "L"  (inicio parcial: _HO -> L)
          " HOL" -> "A"  (casi completo:  _HOL -> A)
          "HOLA" -> " "  ignorado (no predecimos espacio)
    """
    conteo = defaultdict(Counter)
    n      = len(corpus)
    ventana = largo_prefijo + 1

    # Padding: agregar espacios al inicio del corpus para capturar
    # los patrones de inicio de palabra con prefijo relleno
    corpus_padded = (' ' * largo_prefijo) + corpus

    n_padded = len(corpus_padded)
    print(f'  Contando n-gramas orden {largo_prefijo+1} ({n:,} chars)...')

    for i in range(n_padded - ventana):
        prefijo = corpus_padded[i : i + largo_prefijo]
        sucesor = corpus_padded[i + largo_prefijo]

        # No predecir espacio: la prediccion siempre es una letra
        if sucesor == ' ':
            continue

        # Permitir espacios al INICIO del prefijo pero no en medio
        # Ejemplo valido:   "   H" (3 espacios + H)
        # Ejemplo invalido: "H LA" (espacio en medio)
        prefijo_sin_espacios_izq = prefijo.lstrip(' ')
        if ' ' in prefijo_sin_espacios_izq:
            continue

        conteo[prefijo][sucesor] += 1

    # Quedarse con el sucesor mas frecuente por prefijo
    tabla = {}
    for prefijo, sucesores in conteo.items():
        mejor, freq = sucesores.most_common(1)[0]
        if freq >= min_freq:
            tabla[prefijo] = mejor

    # ORDEN LEXICOGRAFICO: obligatorio para busqueda binaria en ESP32
    # El espacio (ASCII 32) va ANTES que las letras (A=65),
    # asi que los prefijos con espacios quedan al inicio de la tabla,
    # exactamente donde la busqueda binaria los buscara.
    tabla_ordenada = dict(sorted(tabla.items()))

    # Contar cuantos prefijos tienen espacios (inicio de palabra)
    con_espacio = sum(1 for k in tabla_ordenada if ' ' in k)
    sin_espacio = len(tabla_ordenada) - con_espacio
    print(f'  -> {len(tabla_ordenada):,} entradas (freq >= {min_freq})')
    print(f'     {con_espacio:,} con espacio (inicio palabra) | {sin_espacio:,} sin espacio')
    return tabla_ordenada


print('OK - funcion construir_tabla lista (con soporte de espacios)')


OK - funcion construir_tabla lista (con soporte de espacios)


---
## Celda 7 — Entrenamiento
Genera los 4 ordenes (T5->T4->T3->T2) para cada idioma.
Esta celda es la mas lenta (~10 min en Colab gratuito).

In [ ]:
print('=' * 55)
print('ENTRENANDO ESPANOL LATINOAMERICANO...')
print('=' * 55)
# T5: prefijo 4 chars -> predice el 5to
tabla_es_T5 = construir_tabla(corpus_es, 4, MIN_FREQ_T5)
# T4: prefijo 3 chars -> predice el 4to
tabla_es_T4 = construir_tabla(corpus_es, 3, MIN_FREQ_T4)
# T3: prefijo 2 chars -> predice el 3ro
tabla_es_T3 = construir_tabla(corpus_es, 2, MIN_FREQ_T3)
# T2: prefijo 1 char  -> predice el 2do
tabla_es_T2 = construir_tabla(corpus_es, 1, MIN_FREQ_T2)

print()
print('=' * 55)
print('ENTRENANDO INGLES USA...')
print('=' * 55)
tabla_en_T5 = construir_tabla(corpus_en, 4, MIN_FREQ_T5)
tabla_en_T4 = construir_tabla(corpus_en, 3, MIN_FREQ_T4)
tabla_en_T3 = construir_tabla(corpus_en, 2, MIN_FREQ_T3)
tabla_en_T2 = construir_tabla(corpus_en, 1, MIN_FREQ_T2)

print()
print('OK - entrenamiento completo')

ENTRENANDO ESPANOL LATINOAMERICANO...
  Contando n-gramas orden 5 (127,399,543 chars)...
  -> 56,248 entradas (freq >= 3)
     5,513 con espacio (inicio palabra) | 50,735 sin espacio
  Contando n-gramas orden 4 (127,399,543 chars)...
  -> 9,136 entradas (freq >= 4)
     616 con espacio (inicio palabra) | 8,520 sin espacio
  Contando n-gramas orden 3 (127,399,543 chars)...
  -> 695 entradas (freq >= 5)
     27 con espacio (inicio palabra) | 668 sin espacio
  Contando n-gramas orden 2 (127,399,543 chars)...
  -> 28 entradas (freq >= 6)
     1 con espacio (inicio palabra) | 27 sin espacio

ENTRENANDO INGLES USA...
  Contando n-gramas orden 5 (78,754,300 chars)...
  -> 55,516 entradas (freq >= 3)
     5,427 con espacio (inicio palabra) | 50,089 sin espacio
  Contando n-gramas orden 4 (78,754,300 chars)...
  -> 9,330 entradas (freq >= 4)
     616 con espacio (inicio palabra) | 8,714 sin espacio
  Contando n-gramas orden 3 (78,754,300 chars)...
  -> 684 entradas (freq >= 5)
     26 con espac

---
## Celda 8 — Funcion de exportacion a .h
Genera el archivo con PROGMEM, structs con sufijo correcto (_latam/_usa)
y predicciones en MAYUSCULAS. El sucesor se guarda como valor decimal
del char para evitar cualquier problema de encoding.

In [ ]:
def escapar_str_c(s):
    """
    Convierte un string Python a representacion segura
    para ponerlo entre comillas dobles en C.
    Espacio -> \\x20
    N con tilde -> \\xD1
    Cualquier otro no-ASCII -> secuencia octal
    """
    out = ''
    for c in s:
        code = ord(c)
        if c == ' ':          out += '\\x20'
        elif c == '"':        out += '\\"'
        elif c == '\\':       out += '\\\\'
        elif c == '\u00D1':   out += '\\xD1'   # N con tilde
        elif 32 <= code <= 126: out += c
        else:                 out += f'\\{code:03o}'
    return out


def exportar_h(tablas, sufijo, descripcion, ruta_salida):
    """
    Exporta las 4 tablas Markov a un archivo .h para Arduino/ESP32.

    Parametros:
        tablas      : dict {'T5':..., 'T4':..., 'T3':..., 'T2':...}
        sufijo      : '_latam' o '_usa'
        descripcion : texto para el comentario de cabecera
        ruta_salida : ruta del archivo .h de salida
    """
    ordenes = ['T5', 'T4', 'T3', 'T2']

    # UTF-8 en Python, pero el contenido del .h es ASCII + escapes
    # asi que el compilador de Arduino lo lee igual sin importar el editor
    with open(ruta_salida, 'w', encoding='utf-8') as f:

        # --- Cabecera solo ASCII ---
        f.write(f'// {"-"*60}\n')
        f.write(f'// {os.path.basename(ruta_salida)}\n')
        f.write(f'// {descripcion}\n')
        f.write(f'// Generado automaticamente - NO EDITAR A MANO\n')
        f.write(f'// Orden N-5 | Cascada T5->T4->T3->T2\n')
        f.write(f'// Tablas ordenadas lexicograficamente (busqueda binaria)\n')
        f.write(f'// {"-"*60}\n')
        f.write('#pragma once\n')
        f.write('#include <Arduino.h>\n\n')

        # --- Structs ---
        # 's' es char (valor directo) para leerlo con pgm_read_byte
        for orden in ordenes:
            f.write(f'struct Entry{orden}{sufijo} {{ const char* k; char s; }};\n')
        f.write('\n')

        # --- Tablas ---
        for orden in ordenes:
            tabla = tablas[orden]
            size  = len(tabla)

            f.write(f'// --- Orden {orden[-1]} | {size:,} entradas ---\n')
            f.write(f'const int SIZE_{orden}{sufijo} = {size};\n')
            f.write(f'const Entry{orden}{sufijo} table_{orden}{sufijo}[] PROGMEM = {{\n')

            for prefijo, sucesor in tabla.items():
                sucesor_may = sucesor.upper()
                k_esc = escapar_str_c(prefijo)
                # Sucesor como valor decimal del char:
                # 'A'=65, 'N con tilde'=209, etc.
                # El ESP32 lo lee con pgm_read_byte y lo castea a char
                s_val = ord(sucesor_may)
                f.write(f'    {"{"}"{k_esc}", {s_val}{"}"}, \n')

            f.write('};\n\n')

    kb = os.path.getsize(ruta_salida) / 1024
    print(f'Exportado: {ruta_salida}  ({kb:.1f} KB)')


print('OK - funcion exportar_h lista')

OK - funcion exportar_h lista


---
## Celda 9 — Exportar los archivos .h

In [ ]:
print('Exportando markov_latam.h...')
exportar_h(
    tablas      = {'T5': tabla_es_T5, 'T4': tabla_es_T4,
                   'T3': tabla_es_T3, 'T2': tabla_es_T2},
    sufijo      = '_latam',
    descripcion = 'Modelo Markov N-5 | Espanol Latinoamericano | Corpus Wikipedia ES',
    ruta_salida = OUT_LATAM
)

print()
print('Exportando markov_usa.h...')
exportar_h(
    tablas      = {'T5': tabla_en_T5, 'T4': tabla_en_T4,
                   'T3': tabla_en_T3, 'T2': tabla_en_T2},
    sufijo      = '_usa',
    descripcion = 'Markov N-5 Model | Neutral US English | Wikipedia EN Corpus',
    ruta_salida = OUT_USA
)

print()
print('Listo. Descarga los archivos desde el panel de archivos de Colab (icono carpeta izquierda).')

Exportando markov_latam.h...
Exportado: /content/markov_latam.h  (1238.1 KB)

Exportando markov_usa.h...
Exportado: /content/markov_usa.h  (1224.7 KB)

Listo. Descarga los archivos desde el panel de archivos de Colab (icono carpeta izquierda).


---
## Celda 10 — Verificacion: prueba de prediccion en Python
Simula exactamente la logica del ESP32 (cascada T5->T4->T3->T2
con prefijo relleno de espacios a la izquierda).
Util para validar antes de flashear.

In [ ]:
# ============================================================
# CELDA 10 CORREGIDA — reemplaza la celda 10 del notebook
# Muestra en que orden de la cascada se encontro la prediccion
# ============================================================

def predecir_verbose(palabra, t5, t4, t3, t2):
    """
    Replica el motor del ESP32 con cascada T5->T4->T3->T2.
    Prefijo relleno con espacios a la izquierda igual que en C.
    Muestra en que nivel de la cascada encontro la prediccion.
    """
    p = palabra.upper()

    # Prefijo de 4 chars relleno con espacios a la izquierda
    p4 = ('    ' + p)[-4:]   # ultimos 4 (con espacios si la palabra es corta)
    p3 = p4[1:]               # ultimos 3
    p2 = p4[2:]               # ultimos 2
    p1 = p4[3:]               # ultimo 1

    if t5.get(p4):
        return t5[p4], 'T5', p4
    if t4.get(p3):
        return t4[p3], 'T4', p3
    if t3.get(p2):
        return t3[p2], 'T3', p2
    if t2.get(p1):
        return t2[p1], 'T2', p1
    return '-', 'MISS', p4


print()
print('=' * 55)
print('TEST ESPANOL LATINO')
print('=' * 55)
for p in ['H', 'HO', 'HOL', 'HOLA',           # inicio de palabra
          'BUENOS', 'COME', 'TAMBI', 'NACION',
          'PRESIDEN', 'QUERI', 'TRABA', 'GOBIER']:
    sig, nivel, prefijo_usado = predecir_verbose(
        p, tabla_es_T5, tabla_es_T4, tabla_es_T3, tabla_es_T2)
    print(f'  "{p:10}" [{nivel}] prefijo="{prefijo_usado}" -> "{sig}"  '
          f'probable: "{p}{sig if sig != "-" else "?"}"')

print()
print('=' * 55)
print('TEST ENGLISH USA')
print('=' * 55)
for p in ['H', 'HE', 'HEL', 'HELL',            # inicio de palabra
          'NATION', 'WORK', 'PRESID', 'THE',
          'COMP', 'SCIEN', 'GOVERN', 'WRITE']:
    sig, nivel, prefijo_usado = predecir_verbose(
        p, tabla_en_T5, tabla_en_T4, tabla_en_T3, tabla_en_T2)
    print(f'  "{p:10}" [{nivel}] prefijo="{prefijo_usado}" -> "{sig}"  '
          f'probable: "{p}{sig if sig != "-" else "?"}"')


TEST ESPANOL LATINO
  "H         " [T3] prefijo=" H" -> "A"  probable: "HA"
  "HO        " [T4] prefijo=" HO" -> "M"  probable: "HOM"
  "HOL       " [T5] prefijo=" HOL" -> "L"  probable: "HOLL"
  "HOLA      " [T5] prefijo="HOLA" -> "N"  probable: "HOLAN"
  "BUENOS    " [T5] prefijo="ENOS" -> "A"  probable: "BUENOSA"
  "COME      " [T5] prefijo="COME" -> "N"  probable: "COMEN"
  "TAMBI     " [T5] prefijo="AMBI" -> "E"  probable: "TAMBIE"
  "NACION    " [T5] prefijo="CION" -> "E"  probable: "NACIONE"
  "PRESIDEN  " [T5] prefijo="IDEN" -> "S"  probable: "PRESIDENS"
  "QUERI     " [T5] prefijo="UERI" -> "A"  probable: "QUERIA"
  "TRABA     " [T5] prefijo="RABA" -> "J"  probable: "TRABAJ"
  "GOBIER    " [T5] prefijo="BIER" -> "N"  probable: "GOBIERN"

TEST ENGLISH USA
  "H         " [T3] prefijo=" H" -> "A"  probable: "HA"
  "HE        " [T4] prefijo=" HE" -> "R"  probable: "HER"
  "HEL       " [T5] prefijo=" HEL" -> "D"  probable: "HELD"
  "HELL      " [T5] prefijo="HELL" -> "E"  probable

---
## Celda 11 — Estadisticas y estimacion de uso de Flash

In [ ]:
def reporte(nombre, t5, t4, t3, t2):
    total = len(t5) + len(t4) + len(t3) + len(t2)
    # Estimacion Flash por entrada:
    # ptr k (4 bytes) + char s (1 byte) + string promedio (5 bytes) = ~10 bytes
    est_kb = total * 10 / 1024
    print(f'\n{nombre}')
    print(f'  T5 (prefijo 4 chars): {len(t5):>8,} entradas')
    print(f'  T4 (prefijo 3 chars): {len(t4):>8,} entradas')
    print(f'  T3 (prefijo 2 chars): {len(t3):>8,} entradas')
    print(f'  T2 (prefijo 1 char) : {len(t2):>8,} entradas')
    print(f'  TOTAL               : {total:>8,} entradas')
    print(f'  Flash estimada      : {est_kb:.0f} KB  ({est_kb/1024:.2f} MB)')

reporte('ESPANOL LATAM', tabla_es_T5, tabla_es_T4, tabla_es_T3, tabla_es_T2)
reporte('ENGLISH USA',   tabla_en_T5, tabla_en_T4, tabla_en_T3, tabla_en_T2)

print()
print('Archivos generados:')
for ruta in [OUT_LATAM, OUT_USA]:
    kb = os.path.getsize(ruta) / 1024
    print(f'  {os.path.basename(ruta)}: {kb:.1f} KB')

print()
print('El ESP32-S3 tiene 8 MB de Flash.')
print('Con PROGMEM los datos van a Flash, NO a RAM (512 KB).')
print('La RAM del programa no se ve afectada.')


ESPANOL LATAM
  T5 (prefijo 4 chars):   56,248 entradas
  T4 (prefijo 3 chars):    9,136 entradas
  T3 (prefijo 2 chars):      695 entradas
  T2 (prefijo 1 char) :       28 entradas
  TOTAL               :   66,107 entradas
  Flash estimada      : 646 KB  (0.63 MB)

ENGLISH USA
  T5 (prefijo 4 chars):   55,516 entradas
  T4 (prefijo 3 chars):    9,330 entradas
  T3 (prefijo 2 chars):      684 entradas
  T2 (prefijo 1 char) :       27 entradas
  TOTAL               :   65,557 entradas
  Flash estimada      : 640 KB  (0.63 MB)

Archivos generados:
  markov_latam.h: 1238.1 KB
  markov_usa.h: 1224.7 KB

El ESP32-S3 tiene 8 MB de Flash.
Con PROGMEM los datos van a Flash, NO a RAM (512 KB).
La RAM del programa no se ve afectada.


---
## Celda 12 — Codigo para el emisor_s3.ino
Copia esta version de `buscarMarkov` en tu `emisor_s3.ino`.
Es necesaria porque los strings `k` estan en PROGMEM y son punteros
que tambien apuntan a PROGMEM. Se requiere `pgm_read_ptr` + `strncpy_P`
para leerlos correctamente. El sucesor `s` se lee con `pgm_read_byte`.

```cpp
template<typename T, int KEY_LEN>
char buscarMarkov(const T* tabla, int size, const char* prefix) {
    int lo = 0, hi = size - 1;
    while (lo <= hi) {
        int mid = (lo + hi) / 2;
        // Leer puntero al string k desde PROGMEM
        const char* k_ptr = (const char*)pgm_read_ptr(&tabla[mid].k);
        // Copiar el string k desde PROGMEM a RAM para comparar
        char key_buf[6] = {0};
        strncpy_P(key_buf, k_ptr, KEY_LEN);
        int cmp = strncmp(prefix, key_buf, KEY_LEN);
        if      (cmp == 0) return (char)pgm_read_byte(&tabla[mid].s);
        else if (cmp  < 0) hi = mid - 1;
        else               lo = mid + 1;
    }
    return 0;
}
```